# Clase 8 — Práctica: el pipeline completo, capstone final

Este es el ejercicio de cierre de todo el curso. Vas a repetir, tú solo, el pipeline completo que viste en el notebook de teoría de esta clase — carga → limpieza → modelado → evaluación — sobre un subconjunto de **cocinas europeas** en vez de asiáticas e india.

El propio notebook oficial de IBM sugiere este ejercicio: *"Feel free to select another subset of cuisines and build a decision tree of their recipes. You can select some European cuisines and build decision trees to explore the ingredients that differentiate those cuisines."* Eso es exactamente lo que vas a hacer, pero esta vez con menos pistas que en cualquier práctica anterior del curso: se espera que apliques por tu cuenta lo que ya sabes de las clases 3, 4, 5 y 8.

Como en las prácticas anteriores, la celda de abajo ya te deja un `recipes` limpio (nombre de columna corregido, minúsculas, nombres de cocina consistentes, cocinas con menos de 50 recetas eliminadas, ingredientes en 1/0). A partir de ahí, el pipeline completo es tu responsabilidad.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import random
from sklearn import tree
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import itertools

# --- Preprocesamiento básico, ya resuelto (igual que en clases anteriores) ---
recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names
recipes["cuisine"] = recipes["cuisine"].str.lower()

recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"
recipes.loc[recipes["cuisine"] == "austria", "cuisine"] = "austrian"
recipes.loc[recipes["cuisine"] == "spain", "cuisine"] = "spanish_portuguese"
recipes.loc[recipes["cuisine"] == "portugal", "cuisine"] = "spanish_portuguese"
recipes.loc[recipes["cuisine"] == "switzerland", "cuisine"] = "swiss"

recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

# Comprueba qué cocinas europeas tienes disponibles en el dataset ya limpio
print(sorted(recipes["cuisine"].unique()))
print("Dataset limpio y listo:", recipes.shape)

## El caso capstone

Eres el data scientist. Tienes `recipes` limpio y listo. Tu tarea: construir y evaluar un árbol de decisión que distinga entre varias **cocinas europeas** (por ejemplo `"french"`, `"italian"`, `"german"`, `"spanish_portuguese"` — revisa la lista impresa arriba, no todos esos nombres tienen por qué existir en el dataset ya filtrado, ajusta tu selección según lo que veas).

Repite, de principio a fin y por tu cuenta, el mismo pipeline que ejecutaste en el notebook de teoría con las cocinas asiáticas e india:

1. **Data Preparation (selección):** filtra `recipes` para quedarte solo con las cocinas europeas que elijas. Separa la variable a predecir (`cuisine`) de las variables de entrada (el resto de columnas, los ingredientes).
2. **Modeling (train/test):** divide en conjunto de entrenamiento y de prueba con `random.seed(1234)` y un `sample_n` razonable por cocina (si alguna cocina europea tiene pocas recetas, puede que 30 sea demasiado — ajusta el valor).
3. **Modeling (entrenamiento):** entrena un `DecisionTreeClassifier` con el conjunto de entrenamiento.
4. **Evaluation:** predice sobre el conjunto de prueba, calcula la matriz de confusión (filas = real, columnas = predicho, en porcentaje) y dibújala, y calcula el accuracy global con `accuracy_score`.

**Pistas generales** (esta vez son más escuetas que en prácticas anteriores — se espera que reconstruyas el resto con lo que ya sabes):

- El patrón de train/test es `groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))` para el conjunto de prueba, y todo lo que no esté en ese índice para el conjunto de entrenamiento.
- `ingredients` son todas las columnas menos la primera (`iloc[:, 1:]`).
- No hay una única "solución correcta": el subconjunto de cocinas que elijas y el `sample_n` que uses son decisiones tuyas, como en cualquier proyecto real.

Trabaja en las celdas de código de abajo. La solución está colapsada justo debajo por si te atascas, pero intenta resolverlo antes de mirarla.

In [ ]:
# TODO: elige entre 3 y 5 cocinas europeas presentes en `recipes` (revisa la lista impresa arriba)
european_cuisines = None

# TODO: filtra `recipes` para quedarte solo con esas cocinas
european_recipes = None

# TODO: separa la variable a predecir (columna "cuisine") de las variables de entrada (el resto de columnas)
cuisines = None
ingredients = None

# TODO: define sample_n (recetas por cocina para el conjunto de prueba) y fija random.seed(1234)
sample_n = None

# TODO: construye el conjunto de prueba y el conjunto de entrenamiento
test_set = None
test_ingredients = None
test_cuisines = None

train_set = None
train_ingredients = None
train_cuisines = None

# TODO: entrena un DecisionTreeClassifier con el conjunto de entrenamiento
euro_tree = None

# TODO: predice las cocinas del conjunto de prueba
euro_pred_cuisines = None

print("Cocinas usadas:", european_cuisines)

In [ ]:
# TODO: calcula la matriz de confusión (test_cuisines vs. euro_pred_cuisines), conviértela a
# porcentaje por fila, y dibújala con plt.imshow (título, colorbar, ticks, valores en cada celda)

labels = None
conf_matrix = None

# TODO: dibuja la matriz de confusión aquí


# TODO: calcula el accuracy global con accuracy_score
accuracy_global = None

print(f"Accuracy global del árbol de cocinas europeas: {accuracy_global:.2%}")

<details>
<summary>💡 Ver solución</summary>

```python
# Elegimos 4 cocinas europeas presentes en el dataset limpio
european_cuisines = ["french", "italian", "german", "spanish_portuguese"]

european_recipes = recipes[recipes.cuisine.isin(european_cuisines)]

cuisines = european_recipes["cuisine"]
ingredients = european_recipes.iloc[:, 1:]

# Usamos un sample_n moderado por si alguna cocina europea tiene pocas recetas
sample_n = 20
random.seed(1234)

test_set = european_recipes.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))
test_ingredients = test_set.iloc[:, 1:]
test_cuisines = test_set["cuisine"]

train_index = european_recipes.index.isin(test_set.index)
train_set = european_recipes[~train_index]
train_ingredients = train_set.iloc[:, 1:]
train_cuisines = train_set["cuisine"]

euro_tree = tree.DecisionTreeClassifier(max_depth=15)
euro_tree.fit(train_ingredients, train_cuisines)

euro_pred_cuisines = euro_tree.predict(test_ingredients)

print("Cocinas usadas:", european_cuisines)

# Matriz de confusión
labels = np.unique(test_cuisines)
conf_matrix = confusion_matrix(test_cuisines, euro_pred_cuisines, labels=labels)
conf_matrix_pct = (conf_matrix.astype("float") / conf_matrix.sum(axis=1)[:, np.newaxis]) * 100

plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix_pct, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Matriz de confusión — cocinas europeas")
plt.colorbar()
tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)

thresh = conf_matrix_pct.max() / 2.0
for i, j in itertools.product(range(conf_matrix_pct.shape[0]), range(conf_matrix_pct.shape[1])):
    plt.text(j, i, format(conf_matrix_pct[i, j], ".2f"),
             horizontalalignment="center",
             color="white" if conf_matrix_pct[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()

# Accuracy global
accuracy_global = accuracy_score(test_cuisines, euro_pred_cuisines)
print(f"Accuracy global del árbol de cocinas europeas: {accuracy_global:.2%}")
```

Nota: como el árbol se entrena con muestreo aleatorio, tus cocinas confundidas entre sí pueden variar respecto a otros compañeros, pero el patrón general (cocinas europeas que comparten muchos ingredientes, como francesa e italiana) debería sostenerse.

</details>

## Reflexión final: cierra el círculo teoría-práctica

Acabas de escribir, tú solo, un pipeline con cuatro bloques de código: selección/filtrado de cocinas, división train/test, entrenamiento del árbol, y evaluación con matriz de confusión y accuracy.

En 2-3 frases, identifica a qué etapa de las 10 de la metodología de Rollins corresponde cada uno de esos cuatro bloques que acabas de escribir, y explica en qué se parece (y en qué no) tu pipeline de cocinas europeas al caso CHF completo que viste en el notebook de teoría de esta clase. Escribe tu respuesta en la celda de abajo.

_Escribe aquí tu reflexión (2-3 frases)._

## Para cerrar el curso

Si completaste este ejercicio, acabas de repetir por tu cuenta, de principio a fin, el mismo tipo de pipeline que un data scientist ejecuta en un proyecto real: cargar datos, limpiarlos, entrenar un modelo, y evaluarlo con criterio — sin que nadie te llevara de la mano en cada línea. Eso es, en la práctica, lo que significa haber aprendido la metodología de ciencia de datos de IBM: no memorizar diez nombres de etapas, sino saber aplicarlas, en orden, a un problema nuevo. Enhorabuena por llegar hasta el final del curso.